# M5b · Backfill `fact_harsh_event`

**Goal:** detect and persist harsh-driving events (brake / accel / corner) from
`staging.archive` accelerometer telemetry into `warehouse.fact_harsh_event`.

**Why this fact exists:** Project 1 (Driver Behavior Scoring & Risk
Classification) requires accelerometer-derived event counts. Without this fact,
the model has no signal for the most predictive risk driver.

**SQL file:** `sql/15_fact_harsh_event_incremental.sql`.
Bound parameters: `:window_start`, `:window_end`, `:etl_run_id`,
`:thresh_brake`, `:thresh_accel`, `:thresh_corner`, `:thresh_high`, `:thresh_extreme`.

**Tunable thresholds** (defaults from `config/pipeline.yaml`
`archive_thresholds`):

| Tier      | Default | Approx g (±2g sensor) |
|-----------|---------|------------------------|
| trigger   | 40      | 0.31 g                 |
| high      | 60      | 0.47 g                 |
| extreme   | 80      | 0.63 g                 |

**Exit criteria:**
- `etl_watermark.fact_harsh_event` advances to ≈ `MAX(staging.archive.date)`.
- Distribution of `event_type` and `severity` is reasonable
  (severe events should be rare; ~1% of all observations).

> **Run order:** before `M6c` (telemetry mart). Independent of fact_trip.


In [1]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


DB = medali_dev@localhost:15432/accent_data
Schemas: staging=staging, warehouse=warehouse, marts=marts


## 2. Inputs — config, thresholds, backfill plan

In [2]:
from accent_fleet.config import load_pipeline_config
from accent_fleet.db.sql_loader import load_sql
from datetime import datetime, timedelta

cfg = load_pipeline_config()
CHUNK_DAYS = cfg['window']['backfill_chunk_days']
MIN_TIME   = datetime.fromisoformat(cfg['window']['min_event_time'].replace('Z','+00:00')).replace(tzinfo=None)
THRESH     = cfg.get('archive_thresholds', {})
print('chunk_days =', CHUNK_DAYS, '  min_time =', MIN_TIME)
print('thresholds =', THRESH)

with get_engine().connect() as conn:
    end_time = conn.execute(text("SELECT MAX(date) FROM staging.archive")).scalar_one()
print('staging.archive max event-time =', end_time)

print('\n----- SQL file preview -----')
print(load_sql('15_fact_harsh_event_incremental.sql')[:800], '...')


chunk_days = 30   min_time = 2019-10-01 00:00:00
thresholds = {'brake': 40, 'accel': 40, 'corner': 40, 'high': 60, 'extreme': 80}
staging.archive max event-time = 2026-04-10 14:11:03

----- SQL file preview -----
-- =============================================================================
-- 15_fact_harsh_event_incremental.sql
-- =============================================================================
-- fact_harsh_event: discrete harsh-driving events derived from staging.archive
-- accelerometer telemetry (x, y, z). One row per detected event.
--
-- Detection rule (per Project 1 spec — Driver Behavior Scoring & Risk
-- Classification): an "event" is a single archive observation whose absolute
-- accelerometer reading on one axis exceeds a calibrated threshold AND whose
-- ignition is on. Severity is bucketed (moderate/high/extreme) so downstream
-- ML can either count totals or weight by severity.
--
-- Axis convention used here (validated by sampling production data):
--   x

## 3. Execute — backfill in chunks

The archive table is large (~55M rows); we process in **smaller** chunks
than the trip facts because each ping does work (UNION ALL × 3 axes).


In [4]:
import importlib, time, pandas as pd
import accent_fleet.transforms.facts as facts_module
from accent_fleet.pipeline.run_log import begin_run, end_run

facts_module = importlib.reload(facts_module)
load_fact_incremental = facts_module.load_fact_incremental

# Override chunk size for the heavy archive table — keep transactions short.
HARSH_CHUNK_DAYS = 7

run_id = begin_run(mode='notebook:06_load_fact_harsh_event')
print('etl_run_id =', run_id)

progress = []
cursor = MIN_TIME
t0 = time.time()
try:
    while cursor < end_time:
        next_cursor = min(cursor + timedelta(days=HARSH_CHUNK_DAYS), end_time)
        res = load_fact_incremental('fact_harsh_event', run_id=run_id, window_end=next_cursor)
        progress.append({'window_start': cursor, 'window_end': next_cursor,
                         'rows_loaded': res.rows_loaded})
        cursor = next_cursor
    end_run(run_id, status='success', rows_loaded=sum(p['rows_loaded'] for p in progress))
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc))
    raise

print(f'done in {time.time()-t0:.1f}s — {len(progress)} chunks')
pd.DataFrame(progress).tail(10)


etl_run_id = 26
2026-04-29 17:33:37 [info     ] fact_load.start                end=datetime.datetime(2019, 10, 8, 0, 0) fact=fact_harsh_event run_id=26 start=datetime.datetime(2019, 10, 1, 0, 0)
2026-04-29 17:33:38 [info     ] fact_load.done                 fact=fact_harsh_event new_watermark=datetime.datetime(2019, 10, 8, 0, 0) rows=0
2026-04-29 17:33:38 [info     ] fact_load.start                end=datetime.datetime(2019, 10, 15, 0, 0) fact=fact_harsh_event run_id=26 start=datetime.datetime(2019, 10, 7, 23, 50)
2026-04-29 17:33:39 [info     ] fact_load.done                 fact=fact_harsh_event new_watermark=datetime.datetime(2019, 10, 15, 0, 0) rows=0
2026-04-29 17:33:39 [info     ] fact_load.start                end=datetime.datetime(2019, 10, 22, 0, 0) fact=fact_harsh_event run_id=26 start=datetime.datetime(2019, 10, 14, 23, 50)
2026-04-29 17:33:39 [info     ] fact_load.done                 fact=fact_harsh_event new_watermark=datetime.datetime(2019, 10, 22, 0, 0) rows=0
2026-04-2

,window_start,window_end,rows_loaded
331,2026-02-03,2026-02-10 00:00:00,84871
332,2026-02-10,2026-02-17 00:00:00,88561
333,2026-02-17,2026-02-24 00:00:00,87225
334,2026-02-24,2026-03-03 00:00:00,78596
335,2026-03-03,2026-03-10 00:00:00,84334
336,2026-03-10,2026-03-17 00:00:00,84680
337,2026-03-17,2026-03-24 00:00:00,55620
338,2026-03-24,2026-03-31 00:00:00,86526
339,2026-03-31,2026-04-07 00:00:00,79502
340,2026-04-07,2026-04-10 14:11:03,50865


## 4. Inspect — watermark, type/severity distribution, sample events

In [5]:
import pandas as pd
with get_engine().connect() as conn:
    wm = pd.read_sql(text("SELECT * FROM warehouse.etl_watermark WHERE table_name='fact_harsh_event'"), conn)
    total = conn.execute(text('SELECT COUNT(*) FROM warehouse.fact_harsh_event')).scalar_one()
    by_type = pd.read_sql(text('''
        SELECT event_type, severity, COUNT(*) AS n
          FROM warehouse.fact_harsh_event
         GROUP BY event_type, severity
         ORDER BY event_type, severity
    '''), conn)
    sample = pd.read_sql(text('''
        SELECT tenant_id, device_id, event_time, event_type, severity,
               x_axis, y_axis, speed_kmh, rpm
          FROM warehouse.fact_harsh_event
         ORDER BY event_time DESC LIMIT 10
    '''), conn)

print(f'fact_harsh_event total rows: {total:,}')
display(wm)
display(by_type)
display(sample)
assert wm['last_event_time'].iloc[0] is not None, 'watermark did not advance'
print('OK — fact_harsh_event backfill complete.')


fact_harsh_event total rows: 1,885,828


,layer,table_name,last_event_time,last_run_at,last_etl_run_id,rows_loaded_total,notes
0,warehouse,fact_harsh_event,2026-04-10 14:11:03,2026-04-29 19:03:52.070709+00:00,26,1886154,Incremental on date (staging.archive); acceler...


,event_type,severity,n
0,harsh_accel,extreme,44039
1,harsh_accel,high,106252
2,harsh_accel,moderate,365817
3,harsh_brake,extreme,27852
4,harsh_brake,high,79684
5,harsh_brake,moderate,331873
6,harsh_corner,extreme,83845
7,harsh_corner,high,234910
8,harsh_corner,moderate,611556


,tenant_id,device_id,event_time,event_type,severity,x_axis,y_axis,speed_kmh,rpm
0,238,427680,2026-04-10 14:10:08,harsh_accel,extreme,109,-34,56,0
1,238,427680,2026-04-10 14:09:08,harsh_accel,extreme,103,-34,87,0
2,264,436348,2026-04-10 14:08:47,harsh_corner,moderate,-17,-43,65,0
3,238,427678,2026-04-10 14:08:45,harsh_brake,moderate,-46,-20,51,1179
4,238,427680,2026-04-10 14:08:08,harsh_accel,extreme,111,-32,85,0
5,238,427680,2026-04-10 14:07:08,harsh_accel,extreme,105,-30,89,0
6,238,427680,2026-04-10 14:06:08,harsh_accel,extreme,119,-47,81,0
7,238,427680,2026-04-10 14:06:08,harsh_corner,moderate,119,-47,81,0
8,238,427607,2026-04-10 14:06:01,harsh_brake,moderate,-56,21,33,0
9,235,425244,2026-04-10 14:05:57,harsh_accel,extreme,83,43,26,1366


OK — fact_harsh_event backfill complete.
